In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

### Question 1

In [2]:
jan_df = pd.read_parquet("./data/yellow_tripdata_2023-01.parquet")
feb_df = pd.read_parquet("./data/yellow_tripdata_2023-02.parquet")

In [3]:
print(
    f"Number of columns in Jan data = {len(jan_df.columns)}"
)

Number of columns in Jan data = 19


### Question 2

In [4]:
jan_df["duration"] = (
    jan_df["tpep_dropoff_datetime"] - jan_df["tpep_pickup_datetime"]
)
jan_df["duration"] = jan_df["duration"].apply(lambda x: x.total_seconds() / 60)
jan_df["duration"].describe()

count    3.066766e+06
mean     1.566900e+01
std      4.259435e+01
min     -2.920000e+01
25%      7.116667e+00
50%      1.151667e+01
75%      1.830000e+01
max      1.002918e+04
Name: duration, dtype: float64

In [5]:
feb_df["duration"] = (
    feb_df["tpep_dropoff_datetime"] - feb_df["tpep_pickup_datetime"]
)
feb_df["duration"] = feb_df["duration"].apply(lambda x: x.total_seconds() / 60)

Standard deviation = 42.59 minutes

### Question 3

In [6]:
filtered_mask = (
    (jan_df["duration"] >= 1) & (jan_df["duration"] <= 60)
)
sum(filtered_mask)

3009173

In [7]:
print(
    f"Percentage of records remaining:"
    + f" {100 * sum(filtered_mask) / len(jan_df):.2f}%"
)

Percentage of records remaining: 98.12%


In [8]:
jan_df = jan_df[filtered_mask]

In [9]:
feb_df = feb_df[(
    (feb_df["duration"] >= 1) & (feb_df["duration"] <= 60)
)]
feb_df["duration"].describe()

count    2.855951e+06
mean     1.446811e+01
std      1.006423e+01
min      1.000000e+00
25%      7.366667e+00
50%      1.181667e+01
75%      1.860000e+01
max      6.000000e+01
Name: duration, dtype: float64

### Question 4

In [10]:
features = ["PULocationID", "DOLocationID"]
target = "duration"

jan_df[features] = jan_df[features].astype(str)
feb_df[features] = feb_df[features].astype(str)

In [11]:
dv = DictVectorizer()

X_train = dv.fit_transform(
    jan_df[features].to_dict(orient="records")
)
X_val = dv.transform(
    feb_df[features].to_dict(orient="records")
)
y_train = jan_df[target]
y_val = feb_df[target]

In [12]:
X_train.shape

(3009173, 515)

Feature count = 515

In [13]:
jan_df["PULocationID"].describe()

count     3009173
unique        255
top           237
freq       147082
Name: PULocationID, dtype: object

In [14]:
jan_df["DOLocationID"].describe()

count     3009173
unique        260
top           236
freq       145424
Name: DOLocationID, dtype: object

In [15]:
print(f"Sum of unique values = {255 + 260}")

Sum of unique values = 515


**Note on One-Hot Encoding and Collinearity:**

Normally, we use $N-1$ features for $N$ categories to avoid the dummy variable trap (perfect multicollinearity). However, `DictVectorizer` creates strict one-hot encoding ($N$ features). We can safely do this in machine learning because:

1. **Robust Solvers**: Scikit-learn's `LinearRegression` uses SVD under the hood, which handles rank-deficient matrices without failing.
2. **Symmetric Regularization Penalty**: If we use regularization (like Ridge or Lasso), it mathematically breaks the perfect collinearity. More importantly, keeping all $N$ columns ensures the penalty is applied symmetrically.
   - *Why is symmetric penalty important?* If you drop Category A, it becomes the "baseline" (absorbed by the intercept), and the weights for Categories B and C represent their difference from A. Regularization would then penalize the weights for B and C, pulling them closer to A, while A gets no explicit penalty. By keeping all $N$ categories, Ridge penalizes all of them evenly towards zero, removing the arbitrary bias of choosing a baseline.

If we explicitly wanted $N-1$ columns, we would use `OneHotEncoder(drop='first')`.

### Question 5

In [16]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_train)
print(f"Train RMSE = {root_mean_squared_error(y_train, y_pred)}")

Train RMSE = 7.649261923759064


### Question 6

In [17]:
y_pred = lr.predict(X_val)
print(f"Val RMSE = {root_mean_squared_error(y_val, y_pred)}")

Val RMSE = 7.811820928728216
